# 1. Import Libraries and architectures
The following PyTorch implementation defines the three architecture variants with dynamic classifier sizing:

In [1]:
import torch
import torch.nn as nn

class ConvBlock(nn.Module):
    """Standard Conv2d + BatchNorm + ReLU + MaxPool2d block."""
    def __init__(self, in_c, out_c):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(in_c, out_c, kernel_size=3, padding=1),
            nn.BatchNorm2d(out_c),
            nn.ReLU(),
            nn.MaxPool2d(2)
        )

    def forward(self, x):
        return self.block(x)


class MNIST_CNN(nn.Module):
    """CNN with configurable depth for concept emergence analysis.

    Args:
        depth: Number of convolutional blocks (2=Shallow, 3=Medium, 5=Deep)
    """
    def __init__(self, depth=3):
        super().__init__()
        self.depth = depth

        # Build convolutional blocks with doubling channels
        layers = []
        c = 1  # Input: 1 channel (grayscale)
        for i in range(depth):
            layers.append(ConvBlock(c, 16 * (2 ** i)))
            c = 16 * (2 ** i)

        self.conv_layers = nn.ModuleList(layers)

        # Compute flattened dimension dynamically
        with torch.no_grad():
            dummy = torch.zeros(1, 1, 28, 28)
            for lyr in self.conv_layers:
                dummy = lyr(dummy)
            flat_dim = dummy.view(1, -1).shape[1]

        # Two-layer FC classifier
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(flat_dim, 128),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(128, 10)  # 10 digit classes
        )

    def forward(self, x):
        for lyr in self.conv_layers:
            x = lyr(x)
        return self.classifier(x)

    def get_activations(self, x, layer_idx):
        """Extract global-average-pooled activations at specified layer.

        Returns: Tensor of shape (batch_size, channels)
        """
        out = x
        for i, lyr in enumerate(self.conv_layers):
            out = lyr(out)
            if i == layer_idx:
                return out.mean(dim=(2, 3))  # GAP: (B, C, H, W) -> (B, C)
        raise ValueError(f'layer_idx {layer_idx} out of range [0, {self.depth-1}]')


# 2. Data Loading and Concept Sets
The concept set definitions are encoded as a lookup table with positive and negative digit class lists:


In [2]:
import torch
from torchvision import datasets, transforms
from torch.utils.data import DataLoader, Subset

# Table 1: Concept definitions (digit class membership)
CONCEPT_MAP = {
    'loop': {
        'pos': [0, 6, 8, 9],
        'neg': [1, 2, 3, 4, 5, 7]
    },
    'vertical_stroke': {
        'pos': [1, 4, 7, 9],
        'neg': [0, 2, 3, 5, 6, 8]
    },
    'horizontal_stroke': {
        'pos': [2, 4, 5, 7],
        'neg': [0, 1, 3, 6, 8, 9]
    },
    'curvature': {
        'pos': [0, 2, 3, 5, 6, 8, 9],
        'neg': [1, 4, 7]
    },
    'intersection': {
        'pos': [4, 8, 9],
        'neg': [0, 1, 2, 3, 5, 6, 7]
    }
}

# Normalisation constants for MNIST (precomputed from training set)
MNIST_MEAN = (0.1307,)
MNIST_STD = (0.3081,)


def get_mnist_loaders(batch_size=128, data_dir='./data'):
    """Create standard MNIST train/test DataLoaders."""
    transform = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize(MNIST_MEAN, MNIST_STD)
    ])

    train = datasets.MNIST(data_dir, train=True, download=True,
                          transform=transform)
    test = datasets.MNIST(data_dir, train=False, download=True,
                         transform=transform)

    train_loader = DataLoader(train, batch_size=batch_size,
                              shuffle=True, num_workers=0)
    test_loader = DataLoader(test, batch_size=batch_size,
                             shuffle=False, num_workers=0)
    return train_loader, test_loader


def get_concept_subsets(dataset, concept_name, max_per_class=500):
    """Create stratified positive/negative subsets for a concept.

    Args:
        dataset: MNIST dataset (train or test)
        concept_name: Key in CONCEPT_MAP
        max_per_class: Maximum samples per digit class

    Returns:
        (positive_subset, negative_subset) as torch.utils.data.Subset
    """
    pos_labels = CONCEPT_MAP[concept_name]['pos']
    neg_labels = CONCEPT_MAP[concept_name]['neg']

    pos_idx = []
    neg_idx = []

    # Collect indices, limiting per-class to maintain balance
    class_counts_pos = {c: 0 for c in pos_labels}
    class_counts_neg = {c: 0 for c in neg_labels}

    for i, (_, y) in enumerate(dataset):
        if y in pos_labels and class_counts_pos[y] < max_per_class:
            pos_idx.append(i)
            class_counts_pos[y] += 1
        elif y in neg_labels and class_counts_neg[y] < max_per_class:
            neg_idx.append(i)
            class_counts_neg[y] += 1

    return Subset(dataset, pos_idx), Subset(dataset, neg_idx)


# 3. Training Pipeline
The training loop includes early stopping, learning rate scheduling, and model checkpointing

In [3]:
import torch
import torch.nn as nn
import torch.optim as optim
from tqdm import tqdm

def train_model(model, train_loader, val_loader,
                epochs=50, lr=1e-3, patience=5,
                weight_decay=1e-4, device='cpu'):
    """Train CNN with early stopping and cosine annealing.

    Returns:
        model: Best model according to validation loss
    """
    model = model.to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=lr,
                           weight_decay=weight_decay)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(
        optimizer, T_max=epochs)

    best_val_loss = float('inf')
    patience_counter = 0
    best_state = None

    for epoch in range(epochs):
        # Training phase
        model.train()
        for x, y in tqdm(train_loader, desc=f'Epoch {epoch+1}'):
            x, y = x.to(device), y.to(device)

            optimizer.zero_grad()
            logits = model(x)
            loss = criterion(logits, y)
            loss.backward()
            optimizer.step()

        scheduler.step()

        # Validation phase
        model.eval()
        val_loss = 0.0
        with torch.no_grad():
            for x, y in val_loader:
                x, y = x.to(device), y.to(device)
                val_loss += criterion(model(x), y).item()
        val_loss /= len(val_loader)

        # Early stopping check
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            # Deep copy state dict to CPU storage
            best_state = {k: v.cpu().clone()
                          for k, v in model.state_dict().items()}
            patience_counter = 0
        else:
            patience_counter += 1
            if patience_counter >= patience:
                print(f'[Early Stop] Epoch {epoch+1} | '
                      f'Val Loss: {val_loss:.4f}')
                break

    # Restore best model
    if best_state is not None:
        model.load_state_dict(best_state)

    return model
